In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import sys, platform, json, importlib.metadata, torch
assert not torch.cuda.is_available(), 'CPU teaching lane unexpectedly sees CUDA'
identity = dict(executable=sys.executable, prefix=sys.prefix, python=platform.python_version(), platform=platform.platform(), cuda_available=torch.cuda.is_available(), torch_cuda_build=torch.version.cuda, packages={name:importlib.metadata.version(name) for name in ['torch','numpy','matplotlib','nbformat','nbclient','ipykernel','transformers','tokenizers','peft']})
print('__DONGXI_KERNEL_IDENTITY__:' + json.dumps(identity, sort_keys=True))

# Three probability distributions, one stored token

Day 20 · Chapter 12 probability bridge; Chapter 14 §14.3 diagnoses the failure. DXI-13, bounded CPU reference, original finite fixtures. No downloaded model, GPU, API or server.

A model gives an action probability 0.30. Collection changes the temperature and discards some actions, so the same action is sampled with probability about 0.2293. Which number belongs in the denominator of its update? The answer depends on **what generated the token**, not on what probabilities are convenient to recompute.

Follow raw model → transformed behavior → stored action/likelihood → declared target objective. The target is a separate choice. We will deliberately break the denominator, then try to ask a truncated sampler about an action it can never observe.

Predict before each complete reference. Change one intervention, rerun and explain the objective you are now optimizing. Material execution is not learner mastery.

[Chapter 12](../../book/chapters/12-language-generation-as-a-policy.md) · [Chapter 14](../../book/chapters/14-when-optimization-goes-wrong.md) · [specification](../../experiments/specs/2026-10-04-sampling-support.md)

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "dongxi_llms").is_dir():
    if root.parent == root:
        raise RuntimeError("Open within Dongxi_LLMs")
    root = root.parent
sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
from dongxi_llms.sampling_likelihood_lab import (
    collect_categorical, target_log_probabilities, importance_ratios,
    importance_expectation, MissingSupportError, sampling_support_experiment,
    kl_gradient_audit, termination_masks, response_surrogate
)
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor":"white", "axes.facecolor":"white",
                     "font.family":"DejaVu Sans", "font.size":11,
                     "axes.spines.top":False, "axes.spines.right":False})
blue, orange, green = "#356a9b", "#b97139", "#3d8268"
raw = torch.tensor([.55, .30, .15], dtype=torch.float64)
rewards = torch.tensor([0., 1., 4.], dtype=torch.float64)
record = collect_categorical(raw.log(), samples=16, seed=2020,
                             temperature=.5, top_k=2, top_p=.9)
print(record.portable())

## Exercise 1 — matching logits do not guarantee matching probabilities

Predict the collected probabilities. The order is temperature → top-k → renormalize → top-p → renormalize. Ties use smaller action IDs first; top-p retains the action crossing the threshold. Our target will deliberately use the **same fixed support but temperature 1**, whereas behavior uses temperature 0.5. Should its target/behavior ratios be one?

A support mask does not encode temperature. Do not call these settings merely display options.

In [ ]:
conditional = target_log_probabilities(raw.log(), 1., record.support).exp()
matched = target_log_probabilities(raw.log(), .5, record.support).exp()
ratios = importance_ratios(conditional, record.probabilities)
torch.testing.assert_close(importance_ratios(matched, record.probabilities)[record.support],
                           torch.ones(2, dtype=torch.float64))
assert not torch.allclose(ratios[record.support], torch.ones(2, dtype=torch.float64))
x = torch.arange(3).numpy()
fig, ax = plt.subplots(figsize=(8,3.8))
for offset, vector, color, label in (
    (-.24, raw, blue, "raw p, temperature 1"),
    (0., record.probabilities, orange, "behavior b, T=.5, truncated"),
    (.24, conditional, green, "target t, T=1, fixed support")):
    ax.bar(x+offset, vector.numpy(), width=.23, color=color, label=label)
ax.set(xticks=x, xticklabels=["action 0","action 1","action 2"],
       ylabel="action probability", ylim=(0,1),
       title="The action ID is unchanged; its probability is not")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print("Target / actual behavior on retained actions:", ratios[record.support].tolist())
print("Matched transformed target / behavior:", importance_ratios(matched, record.probabilities)[record.support].tolist())

**Reference explanation:** raw p is [0.55,0.30,0.15]; behavior retains actions 0/1 and renormalizes their squared probabilities. The temperature-one conditional target renormalizes the original probabilities instead. Only the temperature-0.5 matched target has ratios one. These are different objectives despite identical source logits and support.

![Saved reference probabilities: raw model, transformed behavior and conditional target](../figures/chapter-12/day-20-03_behavior_probabilities_and_support-01.png)

The saved preview is fixed; rerun the preceding plot to reflect edited settings. Read the orange and green bars together, then change temperature only.

## Exercise 2 — a wrong denominator changes value and gradient

For one fixed state, target reward is $J=\sum_a t_\theta(a)R(a)$. With frozen behavior $b$ covering target support, exact importance enumeration is $\sum_a b(a)[t_\theta(a)/b(a)]R(a)$.

Predict whether replacing $b$ in the denominator with raw model p preserves J. Freeze both collection and target support; gradients flow through current target logits, not through the recorded behavior or reward. The analytical conditional-softmax gradient is $t_i(R_i-J)$ on its support and zero outside it. This is an unclipped single-state expectation, not all of PPO.

In [ ]:
result = sampling_support_experiment()
rows = result["expectations"]
direct, corrected, wrong = [rows[k] for k in (
    "direct_conditional_target", "correct_denominator", "wrong_raw_denominator")]
assert abs(direct["value"]-corrected["value"]) < 1e-14
torch.testing.assert_close(torch.tensor(direct["gradient"]), torch.tensor(corrected["gradient"]))
analytic = conditional * (rewards-direct["value"])
torch.testing.assert_close(torch.tensor(direct["gradient"],dtype=torch.float64), analytic)
fig, axes = plt.subplots(1,2,figsize=(10,3.7))
labels = ["direct target","correct b","wrong raw p"]
axes[0].bar(labels,[r["value"] for r in (direct,corrected,wrong)], color=[blue,green,orange])
axes[0].set(ylabel="exact expected reward",title="Same target; wrong collection denominator")
for offset, row, color, label in zip((-.24,0,.24),(direct,corrected,wrong),(blue,green,orange),labels):
    axes[1].bar(x+offset,row["gradient"],width=.23,color=color,label=label)
axes[1].axhline(0,color=".6")
axes[1].set(xticks=x,xticklabels=["0","1","2"],xlabel="target logit index",ylabel="dJ / dz",title="The optimizer sees a different gradient")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
print(rows)

**Reference explanation:** the correct denominator cancels the frozen behavior weights exactly in the finite sum. The wrong denominator leaves an unintended action-dependent multiplier $b/p$. This changes the objective, not only a diagnostic ratio. Both value and gradient reveal the bug. Exact enumeration has no Monte Carlo uncertainty here.

![Saved reference expectation and gradient comparisons](../figures/chapter-12/day-20-03_behavior_probabilities_and_support-02.png)

Predict how moving reward 1 to action 0 changes the wrong-denominator error. The fixed-support repair is a **conditional objective**, not the original full-support objective.

## Exercise 3 — missing support is missing evidence

Now declare the target to be raw full-support p. Action 2 has mass 0.15 and reward 4 but cannot be generated by b. Can a retained mask repair this original objective without changing it? What is the excluded contribution to expected reward?

Answer extraction, group filtering and importance clipping do not recreate unsampled actions.

In [ ]:
try:
    importance_ratios(raw, record.probabilities)
except MissingSupportError as error:
    print("Expected rejection:", error)
else:
    raise AssertionError("An uncovered target cannot be silently estimated")
missing_mass = raw[~record.support].sum()
missing_reward = (raw*rewards)[~record.support].sum()
print("Excluded target probability mass:", float(missing_mass))
print("Excluded expected reward contribution:", float(missing_reward))
print("Full raw target reward:", float((raw*rewards).sum()))
assert abs(float(missing_reward)-.6) < 1e-14
# A legitimate alternative: collect with full support at temperature 1.
full = collect_categorical(raw.log(), temperature=1., top_k=None, top_p=1.)
torch.testing.assert_close(importance_expectation(raw, full.probabilities, rewards), (raw*rewards).sum())

**Reference explanation:** 0.15 probability mass and 0.60 expected reward contribution are invisible to this truncated collector. Zeroing the third ratio would conceal, not solve, the issue. Two explicit alternatives are full-support collection for the raw target, or a declared conditional target on the fixed retained support. The latter changes the objective. A frozen support also avoids silently differentiating through a hard top-k/top-p boundary. Recomputing support after every update defines another, discontinuous procedure.

## Exercise 4 — equal KL estimator values do not imply equal gradients

With raw current p and frozen raw reference q, $D_{\mathrm{KL}}(p\Vert q)=\sum_a p(a)\log[p(a)/q(a)]$.

At a fresh state, k1 is log(p/q) and k3 is q/p − 1 + log(p/q). Both have the correct expectation under p when support is shared. If the sampled action distribution is held fixed at $b=p_0$ during autodiff, should either sample formula automatically reproduce the gradient of the **full** KL? The state distribution here is fixed; we are isolating the action distribution.

Predict before viewing the gradient panel. A value estimate and an optimization surrogate are different contracts.

In [ ]:
reference = torch.tensor([.2,.5,.3], dtype=torch.float64)
audit = kl_gradient_audit(raw.log(), reference.log())
names = list(audit)
values = torch.tensor([audit[n]["value"] for n in names])
gradients = torch.tensor([audit[n]["gradient"] for n in names])
torch.testing.assert_close(values, values[0].expand_as(values))
torch.testing.assert_close(gradients[3],gradients[0])
torch.testing.assert_close(gradients[4],gradients[0])
fig, axes = plt.subplots(1,2,figsize=(11,3.9))
axes[0].bar(range(len(names)),values.numpy(),color=[blue,orange,orange,green,green])
axes[0].set_xticks(range(len(names)),names,rotation=32,ha="right")
axes[0].set(ylabel="expected KL value (nats)",title="All five values agree at the fresh state")
bound = float(gradients.abs().max())
im = axes[1].imshow(gradients.numpy(),cmap="RdBu_r",vmin=-bound,vmax=bound,aspect="auto")
axes[1].set(yticks=range(len(names)),yticklabels=names,xticks=x,xticklabels=["0","1","2"],
            xlabel="current logit index",title="Same value, different autodiff gradient")
for i in range(len(names)):
    for j in range(3):
        axes[1].text(j,i,f"{gradients[i,j]:+.3f}",ha="center",va="center",fontsize=9)
fig.colorbar(im,ax=axes[1],label="dKL / dz")
plt.tight_layout(); plt.show()
print(audit)

**Reference explanation:** holding b fixed omits the derivative of the action distribution. Here frozen k1 has expected gradient zero; frozen k3 has gradient p−q, generally different from the exact forward-KL gradient $p_i[\log(p_i/q_i)-D_{\mathrm{KL}}]$. Differentiating the full-support weight p/b as well as the sample term restores the exact categorical KL gradient in this finite demonstration. That is not a claim about a complete trajectory-KL objective with changing prefix-state distributions.

![Saved reference equal KL values and unequal frozen-sample gradients](../figures/chapter-12/day-20-03_behavior_probabilities_and_support-03.png)

Green rows are explicit importance-weighted identities. Detaching an extra factor may keep a forward value unchanged while changing the update. Always state whether a KL expression is a metric or an intended differentiable loss.

## Exercise 5 — EOS is an action; a cap is a boundary

One row produces EOS; another runs out of its three-token budget. Padding uses the same ID as EOS. Which positions receive policy loss? Which boundary permits bootstrapping for a continuing-task critic? That permission is a task-modeling convention, not a claim that this notebook trains a critic.

Predict the mask. Only generated stop IDs terminate; a stop inside the prompt or padding must not. Loss masking says which actions are scored, not which earlier states can influence them.

In [ ]:
ids = torch.tensor([[2,8,4,2,2,2],[8,9,4,5,6,2]])
boundaries = termination_masks(ids, torch.tensor([2,2]), torch.tensor([4,5]),
                               torch.tensor([3,3]), stop_ids=[2])
mask = boundaries["response_mask"]
assert boundaries["terminal"].tolist() == [True,False]
assert boundaries["truncated"].tolist() == [False,True]
fig, ax = plt.subplots(figsize=(8,2.9))
ax.imshow(mask.numpy(),cmap="Blues",vmin=0,vmax=1,aspect="auto")
for i in range(2):
    for j in range(6):
        ax.text(j,i,str(int(ids[i,j])),ha="center",va="center",
                color="white" if mask[i,j] else "black")
ax.set(xticks=range(6),xlabel="collated token position (not shifted logits)",
       yticks=[0,1],yticklabels=["EOS terminal","token-cap truncation"],
       title="Blue = scored response action; EOS is included, pad is not")
plt.tight_layout(); plt.show()
print({k:v.tolist() for k,v in boundaries.items()})

**Reference explanation:** the first row scores response token 4 and its generated EOS. The second scores all three generated tokens; its EOS-valued padding is not termination. For a continuing-task critic, EOS prevents bootstrap, while a time-limit boundary can permit it if the next state/value is available. A finite-horizon task can instead declare the cap terminal. The mask utility rejects an unexplained early end, and model logit positions must still be shifted/aligned once.

![Saved reference response masks with EOS-as-padding](../figures/chapter-12/day-20-03_behavior_probabilities_and_support-04.png)

## Exercise 6 — identify the gradient boundaries

Current target logits must receive gradients; recorded old/behavior probabilities, advantages and frozen reference weights must not. Padding may contain meaningless likelihood placeholders. Should zero times NaN be trusted to remove their effect?

In [ ]:
theta = torch.zeros(2,6,3,dtype=torch.float64,requires_grad=True)
old = torch.full((2,6),float("nan"),dtype=torch.float64)
old[mask] = -torch.log(torch.tensor(3.,dtype=torch.float64))
old.requires_grad_(True)
advantage = torch.ones(2,6,dtype=torch.float64,requires_grad=True)
ref = torch.zeros_like(theta,requires_grad=True)
actions = torch.ones(2,6,dtype=torch.long)
loss = response_surrogate(theta,actions,old,advantage,mask,
                           reference_logits=ref,beta=.02)
loss.backward()
assert torch.isfinite(theta.grad).all()
assert float(theta.grad[~mask].abs().sum()) == 0.
assert old.grad is None and advantage.grad is None and ref.grad is None
print("Finite current gradients, zero padded gradient; old/advantage/reference detached.")
print("Loss:",float(loss.detach()))

**Reference explanation:** select valid positions before likelihood arithmetic. Multiplying NaN by a zero mask does not remove NaN. Old and reference have different roles: old describes collection, reference defines an anchor. Neither trains through this surrogate. The advantage is a detached signal. Exact reference KL still differentiates through the current target distribution.

## Defense and evidence boundary

Explain which of these changes an objective: temperature, top-p support, the denominator, detaching a factor, and a cap-terminal convention. Repair one declared failure while preserving the original full-support objective; then contrast a conditional-target repair that deliberately changes it.

All observations here concern exact finite probabilities and gradients. Stored draws illustrate record integrity, not an LLM capability experiment. We did not change the course Qwen adapter's temperature-one/full-support baseline. CPU verification on Spark is not Mac verification, PPO reproduction or proof of model-scale stability.

[Worked solutions](../../book/solutions/14-when-optimization-goes-wrong.md#11-three-probability-distributions-and-a-support-boundary) · [recorded CPU evidence](../../experiments/reports/2026-10-04-sampling-support.md)